# Brian LightGBM v3

Restructured Week 3 model for the MLSN portfolio project.

Design commitments:
- Build features once on full history.
- Define the weekly decision calendar before training.
- Train only on rebalance decisions.
- Use LightGBM lambdarank with groups by execution date.
- Use purged walk-forward validation with an embargo at least as large as the prediction horizon.
- Keep NaN feature values for LightGBM instead of blanket zero-imputation.
- Convert model scores into a constrained, risk-aware long-only portfolio.

Decision-time convention:
- Signal date is the previous trading day close.
- Execution date is the first trading day of the next calendar week.
- The supervised target is forward 10-trading-day alpha vs SPY from the signal date.
- Prediction rows are dated by execution date because those rows become portfolio weights.


In [1]:
import os
import sys
from pathlib import Path
import json
import math

import numpy as np
import pandas as pd
import lightgbm as lgb


def is_repo_root(path: Path) -> bool:
    return (path / "pyproject.toml").exists() and (path / "src" / "portfolio_toolkit").exists()


def find_repo_root() -> Path:
    candidates = [Path.cwd()] + list(Path.cwd().parents)
    for path in candidates:
        if is_repo_root(path):
            return path
    raise RuntimeError("Could not find repo root. Run this notebook from inside the repository.")


repo_root = find_repo_root()
os.chdir(repo_root)
src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from portfolio_toolkit import (
    PortfolioWeights,
    backtest_weights,
    baseline_weights,
    build_features,
    build_metrics,
    get_dataset_spec,
    init_mlflow,
    load_prices,
    log_backtest,
    log_model_submission,
    log_portfolio,
    log_predictions,
    log_report_artifacts,
    make_forward_alpha_target,
    split_dates,
    start_run,
    validate_feature_frame,
    validate_prediction_frame,
    validate_weights_frame,
    write_backtest_artifacts,
)

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

print("repo_root =", repo_root)
print("Imports successful.")


repo_root = C:\Users\brixn\Documents\Portfolio-Optimization-Lib
Imports successful.


In [28]:
# Configuration
DATASET_NAME = "shared_set_1"
MODEL_VERSION = "v3"
MODEL_NAME = "Brian_lgbm_v3_lambdarank_weekly_risk"
STRATEGY_NAME = "brian_lgbm_v3_lambdarank_weekly_risk"

HORIZON = 10
EMBARGO_DAYS = HORIZON
REBALANCE_FREQUENCY = "weekly_first_trading_day"
N_LABEL_BINS = 5

TOP_N = 150
MAX_WEIGHT = 0.025
TURNOVER_BLEND = 0.60
RISK_POWER = 0.5
VOL_FLOOR = 0.005

RANDOM_SEED = 42
RUN_MLFLOW = False  # Set True when Tailscale is connected and you want to log the formal run.

output_dir = repo_root / "runs" / "Brian_lgbm_v3"
output_dir.mkdir(parents=True, exist_ok=True)
model_path = repo_root / "MODELS" / "Brian" / "lgbm_v3_model.txt"
metadata_path = repo_root / "MODELS" / "Brian" / "lgbm_v3_metadata.json"
notebook_path = repo_root / "MODELS" / "Brian" / "brian_lgbm_v3.ipynb"

spec = get_dataset_spec(DATASET_NAME, repo_root=repo_root)
SPLITS = split_dates(DATASET_NAME, repo_root=repo_root)
TRAIN_START, TRAIN_END = SPLITS["train"]
VAL_START, VAL_END = SPLITS["val"]
TEST_START, TEST_END = SPLITS["test"]

print("Dataset:", DATASET_NAME, spec.name)
print("Horizon:", HORIZON)
print("Rebalance:", REBALANCE_FREQUENCY)
print("Embargo days:", EMBARGO_DAYS)
print("Official train:", TRAIN_START.date(), "to", TRAIN_END.date())
print("Official val:  ", VAL_START.date(), "to", VAL_END.date())
print("Official test: ", TEST_START.date(), "to", TEST_END.date())


Dataset: shared_set_1 sp500_full_universe
Horizon: 10
Rebalance: weekly_first_trading_day
Embargo days: 10
Official train: 2014-01-02 to 2019-12-31
Official val:   2020-01-02 to 2021-12-31
Official test:  2022-01-03 to 2025-12-31


In [3]:
# Data and calendar
prices = load_prices(DATASET_NAME, repo_root=repo_root)
prices["date"] = pd.to_datetime(prices["date"], utc=True).dt.tz_localize(None)
prices["ticker"] = prices["ticker"].astype(str).str.upper()
prices = prices.sort_values(["ticker", "date"]).reset_index(drop=True)

TRADING_DATES = pd.DatetimeIndex(sorted(prices["date"].unique()))


def trading_day_offset(date_value, offset: int, trading_dates: pd.DatetimeIndex = TRADING_DATES) -> pd.Timestamp:
    date_value = pd.Timestamp(date_value)
    position = int(trading_dates.searchsorted(date_value, side="left"))
    if position >= len(trading_dates):
        position = len(trading_dates) - 1
    target = min(max(position + int(offset), 0), len(trading_dates) - 1)
    return pd.Timestamp(trading_dates[target])


def build_decision_calendar(prices_frame: pd.DataFrame) -> pd.DataFrame:
    dates = pd.DatetimeIndex(sorted(pd.to_datetime(prices_frame["date"], utc=True).dt.tz_localize(None).unique()))
    first_by_week = (
        pd.DataFrame({"date": dates})
        .groupby(pd.Series(dates).dt.to_period("W"), sort=True)["date"]
        .first()
    )
    records = []
    for execution_date in pd.DatetimeIndex(first_by_week.to_numpy()):
        position = dates.searchsorted(execution_date, side="left")
        if position <= 0:
            continue
        records.append({
            "signal_date": pd.Timestamp(dates[position - 1]),
            "date": pd.Timestamp(execution_date),
        })
    return pd.DataFrame(records).drop_duplicates("date").sort_values("date").reset_index(drop=True)


def date_window(frame: pd.DataFrame, start, end, column: str = "date") -> pd.DataFrame:
    return frame.loc[(frame[column] >= pd.Timestamp(start)) & (frame[column] <= pd.Timestamp(end))].copy()


decision_calendar = build_decision_calendar(prices)
decision_calendar = date_window(decision_calendar, TRAIN_START, TEST_END, column="date")

print("Prices loaded:", prices.shape)
print("Price range:", prices["date"].min().date(), "to", prices["date"].max().date())
print("Decision dates:", len(decision_calendar), decision_calendar["date"].min().date(), "to", decision_calendar["date"].max().date())
print(decision_calendar.head().to_string(index=False))


Prices loaded: (1463605, 8)
Price range: 2014-01-02 to 2025-12-31
Decision dates: 626 2014-01-06 to 2025-12-29
signal_date       date
 2014-01-03 2014-01-06
 2014-01-10 2014-01-13
 2014-01-17 2014-01-21
 2014-01-24 2014-01-27
 2014-01-31 2014-02-03


In [4]:
# Feature factory
TOOLKIT_FEATURES = [
    "momentum_5d", "momentum_20d", "momentum_60d", "momentum_120d",
    "vol_20d", "vol_60d", "downside_vol_20d", "upside_vol_20d",
    "beta_20d_spy", "beta_60d_spy",
    "price_to_sma_20d", "price_to_sma_50d", "macd_hist",
    "rsi_14", "bollinger_z_20d",
    "excess_return_5d_vs_spy", "excess_return_20d_vs_spy", "excess_return_60d_vs_spy",
    "relative_momentum_20d_vs_spy",
    "volume_zscore_20d", "volume_zscore_60d", "dollar_volume_ratio_20d",
    "distance_to_20d_high", "distance_to_60d_high",
]

CUSTOM_FEATURES = [
    "mom_vol_ratio",
    "downside_ratio",
    "mom_divergence",
    "mom_60_20_divergence",
    "mom_beta_adjusted",
    "trend_strength",
    "spy_return_20d",
    "spy_vol_20d",
    "spy_drawdown_60d",
]

FEATURE_COLS = TOOLKIT_FEATURES + CUSTOM_FEATURES


def add_custom_features(features: pd.DataFrame, prices_ref: pd.DataFrame) -> pd.DataFrame:
    features = features.copy()
    features["mom_vol_ratio"] = features["momentum_20d"] / features["vol_20d"].replace(0.0, np.nan)
    features["downside_ratio"] = features["downside_vol_20d"] / features["vol_20d"].replace(0.0, np.nan)
    features["mom_divergence"] = features["momentum_5d"] - features["momentum_20d"]
    features["mom_60_20_divergence"] = features["momentum_60d"] - features["momentum_20d"]
    features["mom_beta_adjusted"] = features["momentum_20d"] / features["beta_20d_spy"].replace(0.0, np.nan)
    features["trend_strength"] = features["price_to_sma_20d"] * features["momentum_20d"]

    spy = (
        prices_ref.loc[prices_ref["ticker"].str.upper() == "SPY", ["date", "adj_close"]]
        .sort_values("date")
        .copy()
    )
    spy["spy_return_20d"] = spy["adj_close"].pct_change(20)
    spy["spy_vol_20d"] = spy["adj_close"].pct_change().rolling(20, min_periods=20).std(ddof=0)
    spy["spy_drawdown_60d"] = spy["adj_close"] / spy["adj_close"].rolling(60, min_periods=60).max() - 1.0
    spy = spy[["date", "spy_return_20d", "spy_vol_20d", "spy_drawdown_60d"]]

    features = features.merge(spy, on="date", how="left")
    features[FEATURE_COLS] = features[FEATURE_COLS].replace([np.inf, -np.inf], np.nan)
    return features


def build_model_features(prices: pd.DataFrame) -> pd.DataFrame:
    features = build_features(prices, feature_names=TOOLKIT_FEATURES)
    features = add_custom_features(features, prices)
    return validate_feature_frame(features)


features = build_model_features(prices)
feature_audit = (
    features[FEATURE_COLS]
    .isna()
    .mean()
    .sort_values(ascending=False)
    .rename("missing_rate")
    .reset_index()
    .rename(columns={"index": "feature"})
)

leaky_feature_names = [name for name in FEATURE_COLS if name.startswith("forward_")]
if leaky_feature_names:
    raise ValueError(f"Forward-looking columns found in features: {leaky_feature_names}")

print("Features built on full history:", features.shape)
print("Feature count:", len(FEATURE_COLS))
print("Top missing feature rates:")
print(feature_audit.head(10).to_string(index=False))


Features built on full history: (1463605, 35)
Feature count: 33
Top missing feature rates:
                 feature  missing_rate
           momentum_120d      0.041272
    mom_60_20_divergence      0.020652
            momentum_60d      0.020652
                 vol_60d      0.020652
            beta_60d_spy      0.020652
excess_return_60d_vs_spy      0.020652
       volume_zscore_60d      0.020308
    distance_to_60d_high      0.020308
        spy_drawdown_60d      0.018463
        price_to_sma_50d      0.016871


In [5]:
# Label factory
TARGET_COL = f"forward_alpha_{HORIZON}d_vs_spy"
LABEL_COL = "alpha_rank_label"


def add_cross_sectional_rank_labels(targets: pd.DataFrame) -> pd.DataFrame:
    labeled = targets.copy()
    labeled = labeled[labeled["ticker"].str.upper() != "SPY"].copy()
    labeled["alpha_rank_pct"] = labeled.groupby("date")[TARGET_COL].rank(pct=True, method="average")
    labeled["alpha_zscore"] = labeled.groupby("date")[TARGET_COL].transform(
        lambda values: (values - values.mean()) / (values.std(ddof=0) + 1e-12)
    )
    raw_label = np.floor(labeled["alpha_rank_pct"] * N_LABEL_BINS)
    labeled[LABEL_COL] = raw_label.clip(lower=0, upper=N_LABEL_BINS - 1)
    labeled.loc[labeled["alpha_rank_pct"].isna(), LABEL_COL] = np.nan
    return labeled


targets = make_forward_alpha_target(prices, horizon=HORIZON)
targets = add_cross_sectional_rank_labels(targets)

print("Targets built:", targets.shape)
print("Target column:", TARGET_COL)
print("Label distribution:")
print(targets[LABEL_COL].value_counts(dropna=False).sort_index().to_string())


Targets built: (1460587, 6)
Target column: forward_alpha_10d_vs_spy
Label distribution:
alpha_rank_label
0.0    289602
1.0    291003
2.0    291060
3.0    291004
4.0    292888
NaN      5030


In [6]:
# Join decision calendar, features, and labels
feature_decisions = (
    features
    .rename(columns={"date": "signal_date"})
    .merge(decision_calendar, on="signal_date", how="inner")
)
feature_decisions = feature_decisions[feature_decisions["ticker"].str.upper() != "SPY"].copy()
feature_decisions = feature_decisions.sort_values(["date", "ticker"]).reset_index(drop=True)

model_frame = (
    feature_decisions
    .merge(
        targets.rename(columns={"date": "signal_date"})[["signal_date", "ticker", TARGET_COL, "alpha_rank_pct", "alpha_zscore", LABEL_COL]],
        on=["signal_date", "ticker"],
        how="left",
    )
)
model_frame = model_frame.dropna(subset=[TARGET_COL, LABEL_COL]).copy()
model_frame[LABEL_COL] = model_frame[LABEL_COL].astype(int)
model_frame = model_frame.sort_values(["date", "ticker"]).reset_index(drop=True)

print("Decision feature rows:", feature_decisions.shape)
print("Labeled model rows:", model_frame.shape)
print("Unique execution dates:", model_frame["date"].nunique())
print("Training frequency check: rows are weekly decision dates only.")
print(model_frame[["signal_date", "date", "ticker", TARGET_COL, LABEL_COL]].head().to_string(index=False))


Decision feature rows: (302963, 36)
Labeled model rows: (301957, 40)
Unique execution dates: 624
Training frequency check: rows are weekly decision dates only.
signal_date       date ticker  forward_alpha_10d_vs_spy  alpha_rank_label
 2014-01-03 2014-01-06      A                  0.062484                 4
 2014-01-03 2014-01-06   AAPL                 -0.004674                 1
 2014-01-03 2014-01-06   ABBV                 -0.039349                 0
 2014-01-03 2014-01-06    ABT                  0.021269                 3
 2014-01-03 2014-01-06   ACGL                 -0.033068                 0


In [7]:
# Purged walk-forward split helpers

def purged_train_validation(frame: pd.DataFrame, val_start, val_end, embargo_days: int = EMBARGO_DAYS) -> tuple[pd.DataFrame, pd.DataFrame, pd.Timestamp]:
    val_start = pd.Timestamp(val_start)
    val_end = pd.Timestamp(val_end)
    signal_cutoff = trading_day_offset(val_start, -int(embargo_days), TRADING_DATES)

    train = frame.loc[
        (frame["date"] < val_start)
        & (frame["signal_date"] <= signal_cutoff)
    ].copy()
    val = frame.loc[
        (frame["date"] >= val_start)
        & (frame["date"] <= val_end)
    ].copy()
    return train, val, signal_cutoff


def build_walk_forward_folds(frame: pd.DataFrame) -> list[dict]:
    fold_years = [2018, 2019, 2020, 2021]
    folds = []
    for year in fold_years:
        val_start = max(pd.Timestamp(f"{year}-01-01"), TRAIN_START)
        val_end = min(pd.Timestamp(f"{year}-12-31"), VAL_END)
        if val_start > val_end:
            continue
        train, val, signal_cutoff = purged_train_validation(frame, val_start, val_end)
        if train.empty or val.empty:
            continue
        folds.append({
            "name": f"wf_{year}",
            "val_start": val_start,
            "val_end": val_end,
            "signal_cutoff": signal_cutoff,
            "train": train,
            "val": val,
        })
    return folds


walk_forward_folds = build_walk_forward_folds(model_frame)
for fold in walk_forward_folds:
    print(
        fold["name"],
        "train_rows=", len(fold["train"]),
        "val_rows=", len(fold["val"]),
        "train_dates=", fold["train"]["date"].nunique(),
        "val_dates=", fold["val"]["date"].nunique(),
        "embargo_signal_cutoff=", fold["signal_cutoff"].date(),
    )

final_signal_cutoff = trading_day_offset(TEST_START, -EMBARGO_DAYS, TRADING_DATES)
final_train = model_frame.loc[
    (model_frame["date"] < TEST_START)
    & (model_frame["signal_date"] <= final_signal_cutoff)
].copy()

test_features = feature_decisions.loc[
    (feature_decisions["date"] >= TEST_START)
    & (feature_decisions["date"] <= TEST_END)
].copy()

test_labeled = model_frame.loc[
    (model_frame["date"] >= TEST_START)
    & (model_frame["date"] <= TEST_END)
].copy()

print("Final train rows:", final_train.shape, "through signal", final_signal_cutoff.date())
print("Test scoring rows:", test_features.shape)
print("Test labeled rows for diagnostics:", test_labeled.shape)


wf_2018 train_rows= 96676 val_rows= 25306 train_dates= 207 val_dates= 53 embargo_signal_cutoff= 2017-12-15
wf_2019 train_rows= 121500 val_rows= 25143 train_dates= 259 val_dates= 52 embargo_signal_cutoff= 2018-12-17
wf_2020 train_rows= 146629 val_rows= 25373 train_dates= 311 val_dates= 52 embargo_signal_cutoff= 2019-12-17
wf_2021 train_rows= 171992 val_rows= 25674 train_dates= 363 val_dates= 52 embargo_signal_cutoff= 2020-12-17
Final train rows: (198153, 40) through signal 2021-12-17
Test scoring rows: (104315, 36)
Test labeled rows for diagnostics: (103309, 40)


In [8]:
# LightGBM ranking helpers
PARAMS = {
    "objective": "lambdarank",
    "metric": "ndcg",
    "eval_at": [5, 10, 25],
    "label_gain": [0, 1, 3, 7, 15],
    "learning_rate": 0.03,
    "num_leaves": 31,
    "max_depth": 6,
    "min_data_in_leaf": 150,
    "feature_fraction": 0.75,
    "bagging_fraction": 0.80,
    "bagging_freq": 1,
    "lambda_l1": 0.10,
    "lambda_l2": 1.00,
    "lambdarank_truncation_level": 50,
    "seed": RANDOM_SEED,
    "feature_fraction_seed": RANDOM_SEED,
    "bagging_seed": RANDOM_SEED,
    "verbosity": -1,
}


def prepare_rank_frame(frame: pd.DataFrame) -> tuple[pd.DataFrame, np.ndarray]:
    ordered = frame.sort_values(["date", "ticker"]).reset_index(drop=True)
    group_sizes = ordered.groupby("date", sort=False).size().to_numpy(dtype=np.int32)
    if int(group_sizes.sum()) != len(ordered):
        raise ValueError("LightGBM group sizes do not match row count.")
    return ordered, group_sizes


def make_lgb_dataset(frame: pd.DataFrame) -> tuple[lgb.Dataset, pd.DataFrame]:
    ordered, group_sizes = prepare_rank_frame(frame)
    X = ordered[FEATURE_COLS].replace([np.inf, -np.inf], np.nan)
    y = ordered[LABEL_COL].astype(int)
    dataset = lgb.Dataset(X, label=y, group=group_sizes, free_raw_data=False)
    return dataset, ordered


def train_ranker(train_frame: pd.DataFrame, val_frame: pd.DataFrame | None = None, num_boost_round: int = 1200):
    train_data, ordered_train = make_lgb_dataset(train_frame)
    valid_sets = None
    valid_names = None
    callbacks = [lgb.log_evaluation(period=100)]

    if val_frame is not None and not val_frame.empty:
        val_data, ordered_val = make_lgb_dataset(val_frame)
        valid_sets = [train_data, val_data]
        valid_names = ["train", "valid"]
        callbacks.append(lgb.early_stopping(stopping_rounds=100, first_metric_only=True))
    else:
        ordered_val = None

    model = lgb.train(
        PARAMS,
        train_data,
        num_boost_round=num_boost_round,
        valid_sets=valid_sets,
        valid_names=valid_names,
        callbacks=callbacks,
    )
    return model, ordered_train, ordered_val


def _date_median_fill(values: pd.Series, fallback: float) -> pd.Series:
    median = values[(values > 0) & values.notna()].median()
    if pd.isna(median):
        median = fallback
    return values.fillna(median).clip(lower=VOL_FLOOR)


def make_prediction_frame(model, frame: pd.DataFrame, global_vol_fallback: float | None = None) -> pd.DataFrame:
    scoring = frame.sort_values(["date", "ticker"]).reset_index(drop=True).copy()
    X = scoring[FEATURE_COLS].replace([np.inf, -np.inf], np.nan)
    raw_scores = model.predict(X, num_iteration=getattr(model, "best_iteration", None) or None)

    scoring["model_score"] = raw_scores
    scoring["signal_rank"] = scoring.groupby("date")["model_score"].rank(pct=True, method="average")

    fallback = global_vol_fallback
    if fallback is None:
        valid_vol = scoring["vol_20d"].replace([np.inf, -np.inf], np.nan)
        fallback = valid_vol[(valid_vol > 0) & valid_vol.notna()].median()
    if pd.isna(fallback):
        fallback = 0.02

    scoring["expected_volatility"] = (
        scoring.groupby("date")["vol_20d"]
        .transform(lambda values: _date_median_fill(values.replace([np.inf, -np.inf], np.nan), float(fallback)))
    )

    predictions = scoring[["date", "ticker", "signal_date", "model_score", "signal_rank", "expected_volatility"]].copy()
    predictions["horizon"] = HORIZON
    predictions["expected_return"] = predictions["signal_rank"]
    return predictions[["date", "ticker", "horizon", "expected_return", "expected_volatility", "signal_date", "model_score", "signal_rank"]]


def score_diagnostics(frame: pd.DataFrame, score_col: str = "model_score", target_col: str = TARGET_COL) -> dict[str, float]:
    if target_col not in frame.columns:
        return {}
    work = frame[["date", score_col, target_col]].replace([np.inf, -np.inf], np.nan).dropna()
    rows = []
    for date_value, group in work.groupby("date", sort=True):
        if group[score_col].nunique() < 2 or group[target_col].nunique() < 2:
            continue
        rank = group[score_col].rank(pct=True, method="average")
        top = group.loc[rank >= 0.80, target_col]
        bottom = group.loc[rank <= 0.20, target_col]
        rows.append({
            "date": date_value,
            "ic": group[score_col].corr(group[target_col], method="pearson"),
            "rank_ic": group[score_col].corr(group[target_col], method="spearman"),
            "top_bottom_spread": top.mean() - bottom.mean() if len(top) and len(bottom) else np.nan,
        })
    if not rows:
        return {"mean_ic": np.nan, "mean_rank_ic": np.nan, "mean_top_bottom_spread": np.nan, "scored_dates": 0}
    metrics = pd.DataFrame(rows)
    return {
        "mean_ic": float(metrics["ic"].mean()),
        "std_ic": float(metrics["ic"].std(ddof=0)),
        "mean_rank_ic": float(metrics["rank_ic"].mean()),
        "std_rank_ic": float(metrics["rank_ic"].std(ddof=0)),
        "mean_top_bottom_spread": float(metrics["top_bottom_spread"].mean()),
        "std_top_bottom_spread": float(metrics["top_bottom_spread"].std(ddof=0)),
        "scored_dates": int(metrics["date"].nunique()),
    }


In [9]:
# Walk-forward cross-validation
fold_rows = []
fold_importance_frames = []
fold_best_iterations = []

for fold in walk_forward_folds:
    print("\nTraining", fold["name"])
    fold_model, _, ordered_val = train_ranker(fold["train"], fold["val"], num_boost_round=1200)
    best_iteration = int(fold_model.best_iteration or fold_model.current_iteration())
    fold_best_iterations.append(best_iteration)

    scored_val = make_prediction_frame(fold_model, ordered_val)
    scored_val = scored_val.merge(
        ordered_val[["date", "ticker", TARGET_COL]],
        on=["date", "ticker"],
        how="left",
    )
    diagnostics = score_diagnostics(scored_val)
    diagnostics.update({
        "fold": fold["name"],
        "best_iteration": best_iteration,
        "train_rows": len(fold["train"]),
        "val_rows": len(fold["val"]),
        "train_dates": fold["train"]["date"].nunique(),
        "val_dates": fold["val"]["date"].nunique(),
    })
    fold_rows.append(diagnostics)

    fold_importance_frames.append(pd.DataFrame({
        "fold": fold["name"],
        "feature": FEATURE_COLS,
        "gain": fold_model.feature_importance(importance_type="gain"),
        "split": fold_model.feature_importance(importance_type="split"),
    }))

cv_results = pd.DataFrame(fold_rows)
if not cv_results.empty:
    print("\nWalk-forward results:")
    print(cv_results.to_string(index=False))
    cv_summary = cv_results[["mean_ic", "mean_rank_ic", "mean_top_bottom_spread", "best_iteration"]].agg(["mean", "std"])
    print("\nCV summary:")
    print(cv_summary.to_string())
else:
    cv_summary = pd.DataFrame()
    print("No walk-forward folds were available.")



Training wf_2018
Training until validation scores don't improve for 100 rounds
[100]	train's ndcg@5: 0.79223	train's ndcg@10: 0.713843	train's ndcg@25: 0.610211	valid's ndcg@5: 0.435696	valid's ndcg@10: 0.43424	valid's ndcg@25: 0.420325
Early stopping, best iteration is:
[67]	train's ndcg@5: 0.750668	train's ndcg@10: 0.677282	train's ndcg@25: 0.588798	valid's ndcg@5: 0.452038	valid's ndcg@10: 0.440771	valid's ndcg@25: 0.420111
Evaluated only: ndcg@5

Training wf_2019
Training until validation scores don't improve for 100 rounds
[100]	train's ndcg@5: 0.775037	train's ndcg@10: 0.687402	train's ndcg@25: 0.593569	valid's ndcg@5: 0.478994	valid's ndcg@10: 0.475359	valid's ndcg@25: 0.444743
Early stopping, best iteration is:
[12]	train's ndcg@5: 0.601807	train's ndcg@10: 0.568608	train's ndcg@25: 0.517196	valid's ndcg@5: 0.526791	valid's ndcg@10: 0.49386	valid's ndcg@25: 0.454245
Evaluated only: ndcg@5

Training wf_2020
Training until validation scores don't improve for 100 rounds
[100]	tra

In [10]:
# Final lambdarank model
if fold_best_iterations:
    FINAL_NUM_BOOST_ROUND = int(np.nanmedian(fold_best_iterations))
else:
    FINAL_NUM_BOOST_ROUND = 400
FINAL_NUM_BOOST_ROUND = max(FINAL_NUM_BOOST_ROUND, 50)

print("Final training rows:", len(final_train))
print("Final training dates:", final_train["date"].nunique())
print("Final num_boost_round:", FINAL_NUM_BOOST_ROUND)

final_train_data, ordered_final_train = make_lgb_dataset(final_train)
model = lgb.train(
    PARAMS,
    final_train_data,
    num_boost_round=FINAL_NUM_BOOST_ROUND,
    callbacks=[lgb.log_evaluation(period=100)],
)
print("Final model iterations:", model.current_iteration())


Final training rows: 198153
Final training dates: 416
Final num_boost_round: 50
Final model iterations: 50


In [11]:
# Generate weekly predictions on the test window
vol_fallback = final_train["vol_20d"].replace([np.inf, -np.inf], np.nan)
vol_fallback = vol_fallback[(vol_fallback > 0) & vol_fallback.notna()].median()

predictions = make_prediction_frame(model, test_features, global_vol_fallback=float(vol_fallback))
predictions = validate_prediction_frame(predictions, dataset_name=DATASET_NAME, horizon=HORIZON, repo_root=repo_root)

scored_test_labeled = predictions.merge(
    test_labeled[["date", "ticker", TARGET_COL]],
    on=["date", "ticker"],
    how="left",
)
test_signal_metrics = score_diagnostics(scored_test_labeled)

print("Predictions validated:", predictions.shape)
print("Prediction date range:", predictions["date"].min().date(), "to", predictions["date"].max().date())
print("Rebalance dates:", predictions["date"].nunique())
print("Signal rank range:", predictions["signal_rank"].min(), "to", predictions["signal_rank"].max())
print("Test signal diagnostics:")
print(pd.Series(test_signal_metrics).to_string())


Predictions validated: (104315, 8)
Prediction date range: 2022-01-03 to 2025-12-29
Rebalance dates: 209
Signal rank range: 0.0019880715705765406 to 1.0
Test signal diagnostics:
mean_ic                     0.048999
std_ic                      0.227150
mean_rank_ic                0.020136
std_rank_ic                 0.196131
mean_top_bottom_spread      0.006436
std_top_bottom_spread       0.033130
scored_dates              207.000000


In [25]:
# Portfolio construction: constrained risk-aware long-only weights

def normalize_with_cap(raw_scores: pd.Series, max_weight: float) -> pd.Series:
    scores = raw_scores.astype(float).clip(lower=0.0).replace([np.inf, -np.inf], np.nan).dropna()
    if scores.empty or scores.sum() <= 0.0:
        scores = pd.Series(1.0, index=raw_scores.index)

    weights = scores / scores.sum()
    cap = max(float(max_weight), 1.0 / len(weights))

    for _ in range(50):
        over = weights > cap
        if not over.any():
            break

        excess = float((weights.loc[over] - cap).sum())
        weights.loc[over] = cap

        under = ~over
        under_total = float(weights.loc[under].sum())
        if under_total <= 0.0 or excess <= 1e-12:
            break

        weights.loc[under] = weights.loc[under] + excess * (weights.loc[under] / under_total)

    return weights / weights.sum()


def build_risk_aware_portfolio(
    predictions: pd.DataFrame,
    *,
    dataset_name: str,
    strategy_name: str,
    top_n: int = TOP_N,
    max_weight: float = MAX_WEIGHT,
    turnover_blend: float = TURNOVER_BLEND,
    risk_power: float = RISK_POWER,
) -> PortfolioWeights:
    validated = validate_prediction_frame(predictions, dataset_name=dataset_name, horizon=HORIZON, repo_root=repo_root)

    if "signal_rank" not in validated.columns:
        raise ValueError("predictions must include signal_rank")

    all_tickers = sorted(get_dataset_spec(dataset_name, repo_root=repo_root).tickers)
    global_vol = validated["expected_volatility"].replace([np.inf, -np.inf], np.nan)
    global_vol = global_vol[(global_vol > 0) & global_vol.notna()].median()
    if pd.isna(global_vol):
        global_vol = 0.02

    rows = []
    index = []
    previous = pd.Series(0.0, index=all_tickers)

    for date_value, frame in validated.groupby("date", sort=True):
        working = frame.copy()
        working["vol_for_weight"] = working["expected_volatility"].replace([np.inf, -np.inf], np.nan)

        date_vol = working.loc[
            (working["vol_for_weight"] > 0) & working["vol_for_weight"].notna(),
            "vol_for_weight",
        ].median()
        if pd.isna(date_vol):
            date_vol = global_vol

        working["vol_for_weight"] = working["vol_for_weight"].fillna(date_vol).clip(lower=VOL_FLOOR)
        working["alpha_score"] = (working["signal_rank"] - 0.50).clip(lower=0.0)
        working["risk_score"] = working["alpha_score"] / (working["vol_for_weight"] ** risk_power)

        min_positions = int(math.ceil(1.0 / max_weight))
        choose_n = min(max(top_n, min_positions), len(working))

        selected = working.sort_values(["risk_score", "signal_rank"], ascending=False).head(choose_n).copy()
        allocation_score = selected.set_index("ticker")["risk_score"]

        if allocation_score.sum() <= 0.0:
            allocation_score = selected.set_index("ticker")["signal_rank"]

        selected_weights = normalize_with_cap(allocation_score, max_weight=max_weight)

        row = pd.Series(0.0, index=all_tickers)
        row.loc[selected_weights.index] = selected_weights

        if turnover_blend > 0.0 and previous.sum() > 0.0:
            blended = (1.0 - turnover_blend) * row + turnover_blend * previous

            # Keep only the strongest names after smoothing so positions do not accumulate forever.
            blended = blended.loc[blended > 0.0].sort_values(ascending=False).head(choose_n)

            capped = normalize_with_cap(blended, max_weight=max_weight)
            row = pd.Series(0.0, index=all_tickers)
            row.loc[capped.index] = capped

        row = row / row.sum()
        rows.append(row)
        index.append(pd.Timestamp(date_value))
        previous = row

    weights = pd.DataFrame(rows, index=pd.DatetimeIndex(index))
    weights.index.name = "date"
    weights = validate_weights_frame(weights, dataset_name=dataset_name, repo_root=repo_root)

    return PortfolioWeights(
        weights=weights,
        dataset_name=dataset_name,
        strategy_name=strategy_name,
        metadata={
            "top_n": top_n,
            "max_weight": max_weight,
            "turnover_blend": turnover_blend,
            "risk_power": risk_power,
            "vol_floor": VOL_FLOOR,
            "score_column": "signal_rank",
            "vol_column": "expected_volatility",
            "position_count_rule": "top_n enforced after turnover blend",
        },
    )


portfolio = build_risk_aware_portfolio(
    predictions,
    dataset_name=DATASET_NAME,
    strategy_name=STRATEGY_NAME,
    top_n=TOP_N,
    max_weight=MAX_WEIGHT,
    turnover_blend=TURNOVER_BLEND,
    risk_power=RISK_POWER,
)

validated_weights = validate_weights_frame(portfolio.weights, dataset_name=DATASET_NAME, repo_root=repo_root)

print("Portfolio weights:", validated_weights.shape)
print("Date range:", validated_weights.index.min().date(), "to", validated_weights.index.max().date())
print("Average names held:", (validated_weights > 0).sum(axis=1).mean())
print("Max names held:", (validated_weights > 0).sum(axis=1).max())
print("Max single-name weight:", validated_weights.max(axis=1).max())
print("Portfolio config:", portfolio.metadata)


Portfolio weights: (209, 503)
Date range: 2022-01-03 to 2025-12-29
Average names held: 150.0
Max names held: 150
Max single-name weight: 0.015071058529424496
Portfolio config: {'top_n': 150, 'max_weight': 0.025, 'turnover_blend': 0.6, 'risk_power': 0.5, 'vol_floor': 0.005, 'score_column': 'signal_rank', 'vol_column': 'expected_volatility', 'position_count_rule': 'top_n enforced after turnover blend'}


In [19]:
# Backtest and portfolio-relevant metrics
result = backtest_weights(DATASET_NAME, portfolio, repo_root=repo_root)
artifacts = write_backtest_artifacts(result, output_dir)

print("=== Brian LightGBM v3 ===")
for key in [
    "annual_return",
    "annual_volatility",
    "sharpe",
    "sortino",
    "max_drawdown",
    "average_turnover",
    "benchmark_annual_return",
    "benchmark_sharpe",
    "benchmark_max_drawdown",
    "sharpe_vs_benchmark",
]:
    print(f"{key}: {result.metrics.get(key)}")

baseline_rows = [
    {
        "strategy": STRATEGY_NAME,
        "annual_return": result.metrics.get("annual_return"),
        "sharpe": result.metrics.get("sharpe"),
        "max_drawdown": result.metrics.get("max_drawdown"),
        "average_turnover": result.metrics.get("average_turnover"),
    },
    {
        "strategy": "SPY benchmark",
        "annual_return": result.metrics.get("benchmark_annual_return"),
        "sharpe": result.metrics.get("benchmark_sharpe"),
        "max_drawdown": result.metrics.get("benchmark_max_drawdown"),
        "average_turnover": np.nan,
    },
]

for baseline_name in ["equal_weight", "inverse_volatility", "momentum_20d"]:
    try:
        baseline = baseline_weights(DATASET_NAME, baseline_name, repo_root=repo_root)
        baseline_result = backtest_weights(DATASET_NAME, baseline, repo_root=repo_root)
        baseline_rows.append(
            {
                "strategy": baseline_name,
                "annual_return": baseline_result.metrics.get("annual_return"),
                "sharpe": baseline_result.metrics.get("sharpe"),
                "max_drawdown": baseline_result.metrics.get("max_drawdown"),
                "average_turnover": baseline_result.metrics.get("average_turnover"),
            }
        )
    except Exception as exc:
        print(f"Skipped baseline {baseline_name}: {exc}")

comparison = pd.DataFrame(baseline_rows).sort_values("sharpe", ascending=False)

print("\n=== Baseline comparison ===")
print(comparison.to_string(index=False))

portfolio_metrics = {
    "mean_ic": test_signal_metrics.get("mean_ic"),
    "mean_rank_ic": test_signal_metrics.get("mean_rank_ic"),
    "mean_top_bottom_spread": test_signal_metrics.get("mean_top_bottom_spread"),
    "sharpe": result.metrics.get("sharpe"),
    "annual_return": result.metrics.get("annual_return"),
    "max_drawdown": result.metrics.get("max_drawdown"),
    "average_turnover": result.metrics.get("average_turnover"),
}

print("\n=== Portfolio-relevant metrics ===")
print(pd.Series(portfolio_metrics).to_string())


=== Brian LightGBM v3 ===
annual_return: 0.18734213068180883
annual_volatility: 0.2603638124588819
sharpe: 0.7195398197335696
sortino: 1.0701547017271331
max_drawdown: -0.26273750873466384
average_turnover: 0.10518041958396705
benchmark_annual_return: 0.10831175107810487
benchmark_sharpe: 0.602767030681329
benchmark_max_drawdown: -0.24496387687011656
sharpe_vs_benchmark: 0.11677278905224064


C:\Users\brixn\Documents\Portfolio-Optimization-Lib\src\portfolio_toolkit\baselines.py:35: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  returns = prices.loc[prices["ticker"].isin(tickers)].pivot(index="date", columns="ticker", values="adj_close").sort_index().pct_change()


Skipped baseline momentum_20d: prediction frame contains null expected_return values

=== Baseline comparison ===
                            strategy  annual_return   sharpe  max_drawdown  average_turnover
brian_lgbm_v3_lambdarank_weekly_risk       0.187342 0.719540     -0.262738          0.105180
                        equal_weight       0.112020 0.639491     -0.200594          0.001015
                       SPY benchmark       0.108312 0.602767     -0.244964               NaN
                  inverse_volatility       0.081635 0.517516     -0.181124          0.022253

=== Portfolio-relevant metrics ===
mean_ic                   0.048999
mean_rank_ic              0.020136
mean_top_bottom_spread    0.006436
sharpe                    0.719540
annual_return             0.187342
max_drawdown             -0.262738
average_turnover          0.105180


In [20]:
# Feature importance with fold stability
final_importance = pd.DataFrame({
    "feature": FEATURE_COLS,
    "final_gain": model.feature_importance(importance_type="gain"),
    "final_split": model.feature_importance(importance_type="split"),
})

if fold_importance_frames:
    fold_importance = pd.concat(fold_importance_frames, ignore_index=True)
    stability = (
        fold_importance
        .groupby("feature")
        .agg(
            fold_gain_mean=("gain", "mean"),
            fold_gain_std=("gain", "std"),
            fold_split_mean=("split", "mean"),
        )
        .reset_index()
    )
    importance = final_importance.merge(stability, on="feature", how="left")
else:
    importance = final_importance.copy()

importance = importance.sort_values("final_gain", ascending=False)
print("Top 20 features. Gain is shown with split count and CV-fold stability, not by gain alone.")
print(importance.head(20).to_string(index=False))


Top 20 features. Gain is shown with split count and CV-fold stability, not by gain alone.
                 feature  final_gain  final_split  fold_gain_mean  fold_gain_std  fold_split_mean
                 vol_60d 1878.485700          107     1156.057076     578.537677            95.75
             spy_vol_20d  638.211278          219      289.122024     220.487501           146.25
           momentum_120d  561.932136          135      392.801502     288.614088           124.50
          spy_return_20d  305.639303          119      183.742511     106.426725            99.00
        downside_vol_20d  266.103086           46      189.781519     197.150568            42.50
    mom_60_20_divergence  263.601834           85       87.451115      54.919924            40.00
                 vol_20d  261.397210           36      115.563644      23.046947            34.50
        spy_drawdown_60d  259.533344          104      145.644443     137.212196            84.50
            beta_60d_spy  23

In [21]:
# Required submission function: predict_from_prices
# build_model_features(prices) is defined above and intentionally does not use targets.


def predict_from_prices(model, prices: pd.DataFrame, dates=None, tickers=None) -> pd.DataFrame:
    features = build_model_features(prices)
    calendar = build_decision_calendar(prices)

    scoring_frame = (
        features
        .rename(columns={"date": "signal_date"})
        .merge(calendar, on="signal_date", how="inner")
    )
    scoring_frame = scoring_frame[scoring_frame["ticker"].str.upper() != "SPY"].copy()

    if dates is not None:
        requested_dates = pd.DatetimeIndex(pd.to_datetime(pd.Series(dates), utc=True).dt.tz_localize(None).unique())
        scoring_frame = scoring_frame.loc[scoring_frame["date"].isin(requested_dates)].copy()

    if tickers is not None:
        requested_tickers = [str(ticker).upper() for ticker in tickers]
        scoring_frame = scoring_frame.loc[scoring_frame["ticker"].isin(requested_tickers)].copy()

    predictions = make_prediction_frame(model, scoring_frame)
    return validate_prediction_frame(predictions, horizon=HORIZON)


submission_check = predict_from_prices(model, prices, dates=validated_weights.index, tickers=spec.tickers)
print("Submission function prediction check:", submission_check.shape)
print(submission_check.head().to_string(index=False))


Submission function prediction check: (104315, 8)
      date ticker  horizon  expected_return  expected_volatility signal_date  model_score  signal_rank
2022-01-03      A       10         0.474747             0.013501  2021-12-31    -0.420282     0.474747
2022-01-03   AAPL       10         0.244444             0.018701  2021-12-31    -0.437161     0.244444
2022-01-03   ABBV       10         0.015152             0.009572  2021-12-31    -0.462849     0.015152
2022-01-03   ABNB       10         0.987879             0.032233  2021-12-31     0.397874     0.987879
2022-01-03    ABT       10         0.332323             0.011641  2021-12-31    -0.431249     0.332323


In [22]:
# Save model artifacts and metadata

def to_jsonable(value):
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    if isinstance(value, (pd.Timestamp,)):
        return value.isoformat()
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, pd.DataFrame):
        return json.loads(value.to_json(orient="records", date_format="iso"))
    if isinstance(value, pd.Series):
        return value.to_dict()
    if isinstance(value, dict):
        return {str(key): to_jsonable(val) for key, val in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_jsonable(item) for item in value]
    return value


model.save_model(str(model_path))

metadata = {
    "model_type": "lightgbm",
    "model_version": MODEL_VERSION,
    "model_name": MODEL_NAME,
    "strategy_name": STRATEGY_NAME,
    "dataset": DATASET_NAME,
    "target": f"{TARGET_COL}_cross_sectional_rank_bins",
    "horizon": HORIZON,
    "rebalance_frequency": REBALANCE_FREQUENCY,
    "decision_time_convention": {
        "signal_date": "previous trading day close",
        "execution_date": "first trading day of the next calendar week",
        "prediction_date_column": "execution date",
    },
    "features": FEATURE_COLS,
    "feature_count": len(FEATURE_COLS),
    "missing_value_policy": "feature NaNs left as NaN for LightGBM; portfolio volatility uses date median then global median fallback",
    "objective": PARAMS["objective"],
    "params": PARAMS,
    "final_num_boost_round": FINAL_NUM_BOOST_ROUND,
    "walk_forward_cv": cv_results if not cv_results.empty else [],
    "test_signal_metrics": test_signal_metrics,
    "portfolio_config": portfolio.metadata,
    "backtest_metrics": result.metrics,
    "source_notebook": "MODELS/Brian/brian_lgbm_v3.ipynb",
    "key_changes_from_previous_v3": [
        "build features on full history before splitting",
        "weekly decision calendar uses first trading day of each week",
        "signal date and execution date are explicit",
        "train and validation rows are weekly rebalance decisions only",
        "purged walk-forward validation with horizon-sized embargo",
        "LightGBM lambdarank objective with groups by execution date",
        "no blanket fillna(0.0) for model features",
        "prediction output keeps model_score and signal_rank while expected_return mirrors signal_rank for toolkit compatibility",
        "risk-aware constrained long-only portfolio layer with max-weight and turnover controls",
    ],
}
metadata = to_jsonable(metadata)
metadata_path.write_text(json.dumps(metadata, indent=2, sort_keys=True) + "\n", encoding="utf-8")

print("Model saved:", model_path)
print("Metadata saved:", metadata_path)


Model saved: C:\Users\brixn\Documents\Portfolio-Optimization-Lib\MODELS\Brian\lgbm_v3_model.txt
Metadata saved: C:\Users\brixn\Documents\Portfolio-Optimization-Lib\MODELS\Brian\lgbm_v3_metadata.json


In [27]:
# Optional MLflow logging
if RUN_MLFLOW:
    import mlflow

    init_mlflow(repo_root=repo_root)
    with start_run(
        run_name=MODEL_NAME,
        dataset_name=DATASET_NAME,
        tags={
            "model_type": "lightgbm",
            "model_family": "lightgbm",
            "horizon": str(HORIZON),
            "target": f"{TARGET_COL}_rank_bins",
            "dataset": DATASET_NAME,
            "rebalance_frequency": REBALANCE_FREQUENCY,
            "version": "3",
            "owner": "Brian",
        },
        repo_root=repo_root,
    ):
        mlflow.log_params({
            "model_name": MODEL_NAME,
            "dataset_name": DATASET_NAME,
            "horizon": HORIZON,
            "feature_count": len(FEATURE_COLS),
            "target": f"{TARGET_COL}_rank_bins",
            "rebalance_frequency": REBALANCE_FREQUENCY,
            "objective": PARAMS["objective"],
            "final_num_boost_round": FINAL_NUM_BOOST_ROUND,
            "embargo_days": EMBARGO_DAYS,
            "top_n": TOP_N,
            "max_weight": MAX_WEIGHT,
            "turnover_blend": TURNOVER_BLEND,
            "portfolio_builder": "notebook_risk_aware_constrained_long_only",
        })
        for key, value in portfolio_metrics.items():
            if value is not None and not pd.isna(value):
                mlflow.log_metric(key, float(value))
        if not cv_results.empty:
            for key in ["mean_ic", "mean_rank_ic", "mean_top_bottom_spread"]:
                mlflow.log_metric(f"cv_{key}_mean", float(cv_results[key].mean()))
                mlflow.log_metric(f"cv_{key}_std", float(cv_results[key].std(ddof=0)))

        log_predictions(predictions)
        log_portfolio(portfolio)
        log_backtest(result)
        log_report_artifacts(artifacts)
        manifest = log_model_submission(
            {"lgbm_v3_model": model_path, "lgbm_v3_metadata": metadata_path},
            model_name=MODEL_NAME,
            model_family="lightgbm",
            feature_names=FEATURE_COLS,
            target=f"{TARGET_COL}_rank_bins",
            horizon=HORIZON,
            rebalance_frequency=REBALANCE_FREQUENCY,
            preprocessing={
                "feature_missing_values": "left_as_nan_for_lightgbm",
                "expected_return_contract": "expected_return mirrors signal_rank for toolkit compatibility",
                "portfolio_volatility_imputation": "date median then global median fallback",
                "spy_removed": True,
            },
            model_config={
                **PARAMS,
                "final_num_boost_round": FINAL_NUM_BOOST_ROUND,
                "label_column": LABEL_COL,
                "label_bins": N_LABEL_BINS,
                "decision_calendar": REBALANCE_FREQUENCY,
                "portfolio_builder": "notebook_risk_aware_constrained_long_only",
                "required_functions": ["build_model_features", "predict_from_prices"],
            },
            source_files=[notebook_path],
            notes="v3: lambdarank weekly model with full-history past-only features, purged walk-forward CV, explicit decision timing, and risk-aware constrained allocation.",
        )
    print(json.dumps(manifest, indent=2))
else:
    print("RUN_MLFLOW is False. Set it to True for the formal logged run when Tailscale is connected.")


🏃 View run Brian_lgbm_v3_lambdarank_weekly_risk at: https://adams-macbook-pro.tail5ddc35.ts.net/#/experiments/2/runs/f6467e6b32cf49d0bbc268830d9ecfa5
🧪 View experiment at: https://adams-macbook-pro.tail5ddc35.ts.net/#/experiments/2
{
  "model_name": "Brian_lgbm_v3_lambdarank_weekly_risk",
  "model_family": "lightgbm",
  "target": "forward_alpha_10d_vs_spy_rank_bins",
  "horizon": 10,
  "rebalance_frequency": "weekly_first_trading_day",
  "feature_names": [
    "momentum_5d",
    "momentum_20d",
    "momentum_60d",
    "momentum_120d",
    "vol_20d",
    "vol_60d",
    "downside_vol_20d",
    "upside_vol_20d",
    "beta_20d_spy",
    "beta_60d_spy",
    "price_to_sma_20d",
    "price_to_sma_50d",
    "macd_hist",
    "rsi_14",
    "bollinger_z_20d",
    "excess_return_5d_vs_spy",
    "excess_return_20d_vs_spy",
    "excess_return_60d_vs_spy",
    "relative_momentum_20d_vs_spy",
    "volume_zscore_20d",
    "volume_zscore_60d",
    "dollar_volume_ratio_20d",
    "distance_to_20d_high",
 